# Training Pipeline

Source: [`train.py`](../train.py), [`mpnranker2.py`](../mpnranker2.py)

## Training Flow

```
1. Parse TrainArgs
2. Load datasets (RepoRT IDs or external TSVs) → Data object
3. preprocess(): compute features, graphs, split, standardize
4. Build RankDataset (pairwise + triplet)
5. DataLoader with optional weighted sampling
6. Instantiate MPNranker
7. train() loop — Adam optimizer, MarginRanking + Triplet loss
8. Save model (.pt), data (.pkl), config (.json)
9. repackage_model.py — combine into single .pt for predict.py
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np

## TrainArgs — All Configurable Parameters

In [ ]:
from train import TrainArgs
import inspect

# Print all arguments with their defaults
src = inspect.getsource(TrainArgs)
print(src)

### Key Training Arguments

| Argument | Default | Description |
|---|---|---|
| `--input` | required | RepoRT dataset IDs or external TSV paths |
| `--epochs` | 10 | Number of training epochs |
| `--batch_size` | 128 | Batch size |
| `--learning_rate` | 1e-5 | Adam learning rate |
| `--epsilon` | `10s` | RT difference threshold for soft pair weights |
| `--encoder_size` | 512 | DMPNN hidden layer size |
| `--sizes` | [512, 256, 128] | Ranking head layer sizes |
| `--sizes_sys` | [256, 256] | System×mol preference encoding layers |
| `--sysinfo` | False | Use column information as system features |
| `--columns_use_hsm` | False | Use HSM column parameters |
| `--columns_use_tanaka` | False | Use Tanaka column parameters |
| `--use_ph` | False | Use mobile phase pH |
| `--sample` | False | Use weighted sampling instead of full dataset |
| `--sampling_count` | 500,000 | Number of pairs per epoch when sampling |
| `--save_data` | False | Save data + config alongside model |

## Recommended Training Command

In [ ]:
CMD = """
python train.py \\
  --input <IDs of RepoRT datasets> \\
  --epsilon 10s \\
  --run_name 2-step \\
  --save_data \\
  --batch_size 512 \\
  --epochs 10 \\
  --sysinfo --columns_use_hsm --columns_use_tanaka --use_ph \\
  --repo_root_folder <path to RepoRT> \\
  --clean_data \\
  --encoder_size 512 \\
  --sizes 256 64 \\
  --sizes_sys 256 256 \\
  --pair_step 1 --pair_stop None \\
  --sample --sampling_count 500_000 --no_group_weights \\
  --no_train_acc_all --no_train_acc \\
  --mpn_no_residual_connections_encoder \\
  --no_standardize \\
  --mpn_no_sigmoid_roi
"""
print(CMD)

## `preprocess()` — Data Preparation

In [ ]:
from train import preprocess
print(inspect.getsource(preprocess))

## Train Loop

In [ ]:
from mpnranker2 import train
print(inspect.getsource(train))

### Key training loop details:

- **Optimizer**: Adam with optional `ExponentialLR` scheduler (`--adaptive_learning_rate`)
- **Gradient clipping**: `clip_grad_norm_(params, 5)`
- **Logging**: TensorBoard via `SummaryWriter` for train/val loss and accuracy
- **Early stopping**: configurable patience on val loss (`--early_stopping_patience`)
- **Epoch saves**: `--ep_save` saves checkpoint after each epoch

## Repackaging the Model for Prediction

In [ ]:
# After training, combine model + scalers into a single .pt file
# Run from project root:
# python repackage_model.py 2-step.pt 2-step_predready.pt

# What repackage_model.py does:
import json
src = open('../repackage_model.py').read()
print(src)

## Monitoring Training with TensorBoard

In [ ]:
# Launch TensorBoard (run in terminal)
# tensorboard --logdir runs/

# Or inline in notebook:
# %load_ext tensorboard
# %tensorboard --logdir ../runs/
print('TensorBoard logs are saved in: runs/<run_name>_{train,val,confl}/')

## Evaluation

In [ ]:
EVAL_CMD = """
python evaluate.py \\
  --model <path to trained model> \\
  --test_sets <IDs of RepoRT datasets> \\
  --repo_root_folder <path to RepoRT> \\
  --epsilon 10s
"""
print(EVAL_CMD)